# Conveyor perception

**An end-to-end industrial CV pipeline on a free T4:** real recycling data → trained model → live detection → drift monitoring → triage decisions.

- **Runtime:** Google Colab T4 (free tier, ~12h cap).  
- **Data:** bundled 4-class recycling set (CC BY 4.0).  
- **Model:** YOLO26s, trained in-kernel, cached on re-run.  
- **Goal:** show the loop — train → infer → drift → triage → maintain — on real data, in <5 minutes.


In [ ]:
# --- Cell 1: Runtime + env check ---
import os, sys, json, platform
from pathlib import Path

# --- 1. Colab or local? ---
IN_COLAB = 'google.colab' in sys.modules
print(f'  Runtime: {"Google Colab" if IN_COLAB else "Local (" + platform.node() + ")"}')

# --- 2. Python + key libs (skip import if missing) ---
print(f'  Python: {sys.version.split()[0]}  ({sys.executable.split("/")[-1]})')
for mod in ['numpy', 'torch', 'ultralytics', 'supervision', 'roboflow']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'  {mod:14s} {v}')
    except ImportError:
        print(f'  {mod:14s} — not installed yet (cell 2 will install)')

# --- 3. GPU (or warn if CPU-only) ---
_gpu = 'unknown'
try:
    import torch
    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
except Exception:
    pass
print(f'  GPU:    {_gpu}')

# --- 4. Disk + RAM ---
_disk_free = '?'
try:
    import shutil
    _u = shutil.disk_usage('/')
    _disk_free = f'{_u.free / 1e9:.1f} GB free of {_u.total / 1e9:.1f} GB'
except Exception:
    pass
print(f'  Disk:   {_disk_free}')

# --- 5. Locate the repo (for local runs) — Colab gets cloned by cell 2 ---
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
if not IN_COLAB:
    # Walk up until we find the repo root (contains pyproject.toml)
    while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
        REPO = REPO.parent
print(f'  Repo:   {REPO}{" (will be cloned here by cell 2)" if IN_COLAB else ""}')

# NOTE: colab_session + the state singleton are intentionally NOT used here.
# Cell 1 runs BEFORE the clone (cell 2), so the repo isn't on disk yet — any
# `import colab_session` would crash with ModuleNotFoundError. State init is
# deferred to cell 3, which runs after the clone + install are done. (Aug 22 2026)
print()
print('  ✓ env check done.  Next: cell 2 (clone + install).')


In [ ]:
# --- Cell 2: Install + clone ---
import os, sys, subprocess, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Clone repo (idempotent — skip if pyproject.toml already there) ---
if IN_COLAB:
    if (REPO / 'pyproject.toml').exists():
        print(f'  Repo already at {REPO} (skipping clone)')
    else:
        print(f'  Cloning conveyor-perception -> {REPO}...')
        REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/roniejosephv-star/conveyor-perception.git',
             str(REPO)],
            check=True,
        )
        print('  ✓ cloned.')
else:
    print(f'  Local repo at {REPO} (skipping clone)')

# --- 2. Add REPO + REPO/notebooks to sys.path (colab_session.py lives in notebooks/) ---
for _p in (REPO, REPO / 'notebooks'):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)
print(f'  Path:  {REPO}  +  {REPO}/notebooks')

# --- 3. Install minimal open-source deps (numpy/torch already on Colab) ---
INSTALL = ['ultralytics', 'supervision', 'opencv-python-headless', 'roboflow']
print(f'  Installing: {", ".join(INSTALL)}')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', *INSTALL]
)
print('  ✓ installed.')

# --- 4. Verify the key imports work (smoke test) ---
print('  Verifying:')
for _mod in ['ultralytics', 'supervision', 'roboflow']:
    try:
        _m = importlib.import_module(_mod)
        _v = getattr(_m, '__version__', '?')
        print(f'    {_mod:14s} {_v}  ok')
    except ImportError as _e:
        print(f'    {_mod:14s} FAIL  {_e}')

# --- 5. Verify colab_session is now importable (the import cell 3 needs) ---
try:
    from colab_session import get_state
    print('    colab_session  ok  (get_state() ready for cell 3)')
except ImportError as _e:
    print(f'    colab_session  FAIL  {_e}')

# --- 6. Quick sanity check: list the repo top-level ---
print(f'  Repo top-level:')
for _entry in sorted(REPO.iterdir()):
    if not _entry.name.startswith('.'):
        print(f'    {_entry.name}')

print()
print('  ✓ install + clone done.  Next: cell 3 (state + toggles).')


In [ ]:
# --- Cell 3: State + toggles ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Defensive install: ipywidgets is needed for the toggle UI ---
# (Colab has it pre-installed, but cell 3 should not assume cell 2 installed it.)
try:
    import ipywidgets  # noqa: F401
    print('  ipywidgets: ok')
except ImportError:
    print('  Installing ipywidgets (toggle UI dep)...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', 'ipywidgets']
    )
    print('  ✓ installed.')

# --- 2. Idempotent sys.path setup (cell 2 already did this; redo is harmless) ---
for _p in (REPO, REPO / 'notebooks'):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 3. Create the SessionState singleton (lives in builtins so every cell sees it) ---
from colab_session import get_state, toggle_ui

state = get_state()
state.log('cell-3', action='init', note='state singleton + toggle UI ready')

# --- 4. Show the toggle UI (4 framework abstractions + 8 JD modules) ---
# Untick anything you want the pipeline to skip. Defaults: all enabled.
print()
print('  Toggle the components below — defaults are all enabled.')
print('  Untick anything you want to skip in the pipeline.')
print()
ui = toggle_ui()
display(ui)

# --- 5. Print a summary of what's currently enabled (grouped) ---
print()
print('  Current toggles:')
abstr_keys = [k for k in state.toggles if k.startswith('abstraction:')]
mod_keys = [k for k in state.toggles if k.startswith('module:')]
print('    4 framework abstractions:')
for _k in abstr_keys:
    _v = state.toggles[_k]
    _icon = '✓' if _v else '○'
    print(f'      {_icon} {_k}')
print('    8 JD modules:')
for _k in mod_keys:
    _v = state.toggles[_k]
    _icon = '✓' if _v else '○'
    print(f'      {_icon} {_k}')

_n_on = sum(state.toggles.values())
_n_total = len(state.toggles)
print()
print(f'  ✓ state + toggles ready.  {_n_on}/{_n_total} components enabled.')
print('  Next: cell 4 (load abstractions).')


In [ ]:
# --- Cell 4: Load abstractions ---
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# --- 1. Path setup: src/ (for `import conveyor_perception`) + REPO/notebooks ---
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 2. Import the 4 abstraction classes (unconditional — fail fast on missing module) ---
from colab_session import get_state
from conveyor_perception.core.detection_pipeline import DetectionPipeline as Detector
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.triage_surface import MCPTriageSurface, InMemoryAlertQueue

state = get_state()
loaded: dict = {}
skipped: list = []

# --- 3. Toggle-gated load (the toggle UI from cell 3 controls this) ---
if state.toggles.get('abstraction:detector', True):
    # Detector is just a class ref for now — model is wired in cell 8 after training.
    loaded['detector'] = Detector
    print('  ✓ Detector class loaded (YOLO26 + OpenCV DNN)')
else:
    skipped.append('abstraction:detector')
    print('  ○ Detector skipped (toggle off)')

if state.toggles.get('abstraction:tracker', True):
    loaded['tracker'] = TrackingPipeline()
    print('  ✓ TrackingPipeline instantiated (ByteTrack)')
else:
    skipped.append('abstraction:tracker')
    print('  ○ TrackingPipeline skipped (toggle off)')

if state.toggles.get('abstraction:drift_monitor', True):
    loaded['drift_monitor'] = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
    print('  ✓ DriftMonitor instantiated (KS test + z-score + MAD)')
else:
    skipped.append('abstraction:drift_monitor')
    print('  ○ DriftMonitor skipped (toggle off)')

if state.toggles.get('abstraction:triage', True):
    loaded['triage_surface'] = MCPTriageSurface('l1-triage', InMemoryAlertQueue())
    print('  ✓ MCPTriageSurface instantiated (5 MCP tools)')
else:
    skipped.append('abstraction:triage')
    print('  ○ MCPTriageSurface skipped (toggle off)')

state.log('cell-4', action='load-abstractions', loaded=list(loaded.keys()), skipped=skipped)
print()
print(f'  ✓ loaded {len(loaded)}/4 abstractions, skipped {len(skipped)}.')
print('  Next: cell 5 (load modules).')


In [ ]:
# --- Cell 5: Load modules ---
import os, sys, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# --- 1. Path setup (idempotent — cell 4 already added REPO/src) ---
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 2. Module metadata: toggle key + import path + 1-line description ---
from colab_session import get_state
state = get_state()

MODULES_META = [
    ('module:perception',             'conveyor_perception.perception',             'UltralyticsDetector + RecyclingInferenceService'),
    ('module:triage',                 'conveyor_perception.triage',                 'L1TriageAgent + 7 severity rules'),
    ('module:predictive_maintenance', 'conveyor_perception.predictive_maintenance', 'MaintenanceAdvisor + 3 signal types'),
    ('module:multitask',              'conveyor_perception.multitask',              'MultitaskPipeline (Detector->Tracker->Drift->Triage)'),
    ('module:integration',            'conveyor_perception.integration',            'ConveyorNode (real ROS 2) + MockROS2Node (CI)'),
    ('module:robustness',             'conveyor_perception.robustness',             'RobustnessTestSuite + 13 augmentations'),
    ('module:monitoring',             'conveyor_perception.monitoring',             'MonitoringDashboard + ShiftReport'),
    ('module:optimization',           'conveyor_perception.optimization',           'benchmark_pytorch/onnx + export_onnx'),
]

loaded: list = []
skipped: list = []
failed: list = []

# --- 3. Toggle-gated dynamic load (one bad module doesn't kill the cell) ---
print('  Loading 8 JD modules:')
for _toggle_key, _module_path, _desc in MODULES_META:
    if not state.toggles.get(_toggle_key, True):
        skipped.append(_toggle_key)
        print(f'    ○ {_module_path}  (disabled by toggle)')
        continue
    try:
        importlib.import_module(_module_path)
        loaded.append(_module_path)
        print(f'    ✓ {_module_path}')
        print(f'        {_desc}')
    except Exception as _e:
        failed.append((_module_path, str(_e)))
        print(f'    ✗ {_module_path}  failed: {_e}')

state.log(
    'cell-5',
    action='load-modules',
    loaded=loaded,
    skipped=skipped,
    failed=[m for m, _ in failed],
)
print()
print(f'  ✓ loaded {len(loaded)}/{len(MODULES_META)} modules, skipped {len(skipped)}, failed {len(failed)}.')
print('  Next: cell 6 (data registry).')
